# Cypher Basics: Querying Graph Data

## Overview

Learn to query your graph data using Cypher, the standard graph query language. This notebook covers schema inspection, basic queries, parameters, result conversion, and exports.

**What you'll learn:**
- Inspect graph schema (node labels, properties, relationships)
- Write Cypher queries with `query()`, `query_scalar()`, `query_df()`
- Use parameters and filters in queries
- Convert results to DataFrames, dictionaries, and NetworkX graphs
- Export results to CSV and Parquet files

**Prerequisites:**
- Completed: [02_core_managing_resources.ipynb](./02_core_managing_resources.ipynb)
- Running instance with data loaded

**Time to complete:** 15 minutes

---

**Test Coverage:**
- Schema inspection (node labels, properties, relationships)
- Query methods (query, query_scalar, query_df, query_one)
- Query features (parameters, filters, paths, aggregations)
- Result iteration, conversion, and file export

In [ ]:
import os

# Parameters cell - papermill will inject values here
EXPECTED_NODE_COUNT = 5
EXPECTED_EDGE_COUNT = 6
INSTANCE_ID = None  # If provided, reuse shared instance (Phase 1.1 optimization)

# Note: Auth is handled via TestPersona - env vars like GRAPH_OLAP_API_KEY_ANALYST_ALICE

## Setup

### Test Data Isolation Strategy

This notebook is **read-only** - it only queries existing data without creating or modifying resources.

In [ ]:
import os
import sys

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")
print(f"Using Bearer token: {'yes' if os.environ.get('GRAPH_OLAP_API_KEY') else 'no'}")
print(f"Expected: {EXPECTED_NODE_COUNT} nodes, {EXPECTED_EDGE_COUNT} edges")

In [ ]:
from graph_olap import notebook
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

# Create test context with automatic cleanup
ctx = notebook.test("QueryTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

# Phase 1.1 Optimization: Reuse shared instance if provided
if INSTANCE_ID is not None:
    print(f"Using shared read-only instance: {INSTANCE_ID}")
    print("  - Skipping mapping/snapshot/instance creation (Phase 1.1 optimization)")
    print("  - Estimated time saved: ~150 seconds")
    
    # Connect to shared instance
    conn = client.instances.connect(int(INSTANCE_ID))
    print(f"Connected to shared instance {INSTANCE_ID}")
else:
    # Standard path: Create resources using ctx (auto-tracked)
    # Define test data
    person_node = NodeDefinition(
        label="Person",
        sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
        primary_key={"name": "id", "type": "STRING"},
        properties=[PropertyDefinition(name="name", type="STRING"), PropertyDefinition(name="age", type="INT64")]
    )
    
    knows_edge = EdgeDefinition(
        type="KNOWS",
        from_node="Person",
        to_node="Person",
        sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
        from_key="from_id",
        to_key="to_id",
        properties=[PropertyDefinition(name="since", type="INT64")],
    )
    
    # Create mapping using ctx (auto-tracked)
    mapping = ctx.mapping(
        description="Mapping for query testing",
        node_definitions=[person_node],
        edge_definitions=[knows_edge],
    )
    MAPPING_ID = mapping.id
    print(f"Created mapping: {mapping.name} (id={MAPPING_ID})")
    
    # SNAPSHOT FUNCTIONALITY DISABLED
    # Create instance directly from mapping using create_from_mapping()
    # The snapshot is created automatically in the background
    print(f"Creating instance directly from mapping (snapshot created automatically)...")
    instance = client.instances.create_from_mapping_and_wait(
        mapping_id=MAPPING_ID,
        name=f"QueryTest-Instance-{ctx.run_id}",
        wrapper_type=WrapperType.RYUGRAPH,
        timeout=300,  # Longer timeout for snapshot export
        poll_interval=5,
    )
    INSTANCE_ID = instance.id
    SNAPSHOT_ID = instance.snapshot_id
    ctx._track('instance', INSTANCE_ID, instance.name)
    print(f"Created instance: (id={INSTANCE_ID}, status={instance.status})")
    print(f"Auto-created snapshot: (id={SNAPSHOT_ID})")
    
    # Connect via client
    conn = client.instances.connect(INSTANCE_ID)
    print(f"Connected to instance {INSTANCE_ID}")

## 1. Schema Inspection Tests

In [ ]:
# Test: Get schema
schema = conn.get_schema()

assert schema is not None, "Schema should not be None"
assert schema.node_labels is not None, "Schema should have node_labels"
assert schema.relationship_types is not None, "Schema should have relationship_types"

print(f"Query 1.1 PASSED: Schema - Nodes: {list(schema.node_labels.keys())}")
print(f"  Edges: {list(schema.relationship_types.keys())}")

In [ ]:
# Test: Schema contains Person node with properties
assert "Person" in schema.node_labels, f"Expected 'Person' in node labels, got {list(schema.node_labels.keys())}"

person_props = schema.node_labels["Person"]
assert "name" in person_props, f"Person should have 'name' property, got {person_props}"
assert "age" in person_props, f"Person should have 'age' property, got {person_props}"

print(f"Query 1.2 PASSED: Person properties: {person_props}")

In [ ]:
# Test: Schema contains KNOWS relationship
assert "KNOWS" in schema.relationship_types, \
    f"Expected 'KNOWS' in relationships, got {list(schema.relationship_types.keys())}"

print(f"Query 1.3 PASSED: KNOWS properties: {schema.relationship_types['KNOWS']}")

## 2. Basic Query Tests

In [ ]:
# Test: query() returns QueryResult
result = conn.query("MATCH (p:Person) RETURN count(p) AS count")

assert result.columns == ["count"], f"Expected ['count'], got {result.columns}"
assert result.row_count == 1, f"Expected 1 row, got {result.row_count}"
assert result.rows[0][0] == EXPECTED_NODE_COUNT, f"Expected {EXPECTED_NODE_COUNT} nodes, got {result.rows[0][0]}"

print(f"Query 2.1 PASSED: Query result: {result.rows[0][0]} nodes")

In [ ]:
# Test: query_scalar returns single value
node_count = conn.query_scalar("MATCH (n) RETURN count(n)")

assert node_count == EXPECTED_NODE_COUNT, f"Expected {EXPECTED_NODE_COUNT} nodes, got {node_count}"
print(f"Query 2.2 PASSED: Node count: {node_count}")

In [ ]:
# Test: Count edges
edge_count = conn.query_scalar("MATCH ()-[r:KNOWS]->() RETURN count(r)")

assert edge_count == EXPECTED_EDGE_COUNT, f"Expected {EXPECTED_EDGE_COUNT} edges, got {edge_count}"
print(f"Query 2.3 PASSED: Edge count: {edge_count}")

## 3. Query with Filters and Parameters

In [ ]:
# Test: Query with filter (WHERE clause)
result = conn.query(
    "MATCH (p:Person) WHERE p.age > 30 RETURN p.name ORDER BY p.name"
)

# Charlie (35) and Eve (32) are over 30
assert result.row_count == 2, f"Expected 2 rows, got {result.row_count}"
names = [row[0] for row in result.rows]
assert "Charlie" in names, "Charlie should be in results"
assert "Eve" in names, "Eve should be in results"

print(f"Query 3.1 PASSED: People over 30: {names}")

In [ ]:
# Test: Query with single parameter
result = conn.query(
    "MATCH (p:Person {name: $name}) RETURN p.age",
    parameters={"name": "Alice"},
)

assert result.row_count == 1, f"Expected 1 row, got {result.row_count}"
assert result.rows[0][0] == 30, f"Expected Alice's age 30, got {result.rows[0][0]}"

print(f"Query 3.2 PASSED: Alice's age: {result.rows[0][0]}")

In [ ]:
# Test: Query with multiple parameters
result = conn.query(
    "MATCH (p:Person) WHERE p.age >= $min_age AND p.age <= $max_age RETURN p.name",
    parameters={"min_age": 28, "max_age": 32},
)

# Alice (30), Diana (28), Eve (32)
assert result.row_count == 3, f"Expected 3 rows, got {result.row_count}"
names = [row[0] for row in result.rows]
assert "Alice" in names and "Diana" in names and "Eve" in names

print(f"Query 3.3 PASSED: People aged 28-32: {names}")

## 4. DataFrame and Result Conversion Tests

In [ ]:
# Test: query_df returns DataFrame
df = conn.query_df("MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY age DESC")

assert len(df) == EXPECTED_NODE_COUNT, f"Expected {EXPECTED_NODE_COUNT} rows, got {len(df)}"
assert "name" in df.columns, "DataFrame should have 'name' column"
assert "age" in df.columns, "DataFrame should have 'age' column"

# Verify sorted descending
ages = df["age"].to_list()
assert ages == sorted(ages, reverse=True), "Ages should be sorted descending"

print(f"Query 4.1 PASSED: DataFrame shape: {df.shape}")
print(df)

In [ ]:
# Test: query_one returns single row as dict
row = conn.query_one("MATCH (p:Person) WHERE p.name = 'Alice' RETURN p.name AS name, p.age AS age")

assert row is not None, "query_one should return a row"
assert row['name'] == 'Alice', f"Expected 'Alice', got '{row['name']}'"
assert isinstance(row['age'], int), f"Age should be int, got {type(row['age'])}"

print(f"Query 4.2 PASSED: query_one result: {row}")

In [ ]:
# Test: query_one returns None for no results
row = conn.query_one("MATCH (p:Person) WHERE p.name = 'NonExistent' RETURN p.name")

assert row is None, f"query_one should return None, got {row}"
print("Query 4.3 PASSED: query_one correctly returns None for no results")

In [ ]:
# Test: Result iteration yields dicts
result = conn.query("MATCH (p:Person) RETURN p.name AS name ORDER BY p.name")

names = []
for row in result:
    names.append(row["name"])

assert len(names) == EXPECTED_NODE_COUNT, f"Expected {EXPECTED_NODE_COUNT} names, got {len(names)}"
assert names == sorted(names), "Names should be sorted alphabetically"

print(f"Query 4.4 PASSED: Iterated names: {names}")

In [ ]:
# Test: Result to_list_of_dicts()
result = conn.query("MATCH (p:Person) RETURN p.name AS name, p.age AS age LIMIT 2")

dicts = result.to_list_of_dicts()
assert len(dicts) == 2, f"Expected 2 dicts, got {len(dicts)}"
assert "name" in dicts[0], "Dict should have 'name' key"
assert "age" in dicts[0], "Dict should have 'age' key"

print(f"Query 4.5 PASSED: to_list_of_dicts: {dicts}")

In [ ]:
# Test: to_networkx() conversion for graph data
# Query graph data with nodes and edges
result = conn.query("""
    MATCH (a:Person)-[r:KNOWS]->(b:Person)
    RETURN a.name AS from_name, b.name AS to_name, r.since AS since
    LIMIT 5
""")

# Convert to NetworkX graph
import networkx as nx

G = result.to_networkx()

assert isinstance(G, (nx.Graph, nx.DiGraph)), f"Expected NetworkX graph, got {type(G)}"
# The graph should have nodes and edges based on the query result
print(f"Query 4.6 PASSED: to_networkx() - {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Test: to_csv() export
import tempfile
import os

result = conn.query("MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name")

# Export to CSV file
with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as f:
    csv_path = f.name

try:
    result.to_csv(csv_path)
    
    # Verify file was created and has content
    assert os.path.exists(csv_path), "CSV file should exist"
    file_size = os.path.getsize(csv_path)
    assert file_size > 0, "CSV file should not be empty"
    
    # Verify content
    with open(csv_path, 'r') as f:
        lines = f.readlines()
    assert len(lines) == EXPECTED_NODE_COUNT + 1, f"CSV should have header + {EXPECTED_NODE_COUNT} rows"
    assert "name" in lines[0] and "age" in lines[0], "CSV header should contain column names"
    
    print(f"Query 4.7 PASSED: to_csv() - exported {file_size} bytes, {len(lines)} lines")
finally:
    os.unlink(csv_path)  # Clean up temp file

In [ ]:
# Test: to_parquet() export
import tempfile
import os

result = conn.query("MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name")

# Export to Parquet file
with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as f:
    parquet_path = f.name

try:
    result.to_parquet(parquet_path)
    
    # Verify file was created and has content
    assert os.path.exists(parquet_path), "Parquet file should exist"
    file_size = os.path.getsize(parquet_path)
    assert file_size > 0, "Parquet file should not be empty"
    
    # Verify we can read it back with polars
    import polars as pl
    df_read = pl.read_parquet(parquet_path)
    assert len(df_read) == EXPECTED_NODE_COUNT, f"Parquet should have {EXPECTED_NODE_COUNT} rows"
    assert "name" in df_read.columns, "Parquet should have 'name' column"
    assert "age" in df_read.columns, "Parquet should have 'age' column"
    
    print(f"Query 4.8 PASSED: to_parquet() - exported {file_size} bytes, {len(df_read)} rows")
finally:
    os.unlink(parquet_path)  # Clean up temp file

In [ ]:
# Test: QueryResult.show() - auto-visualization method
# Note: show() is designed for Jupyter display, we verify it doesn't crash and returns something
result = conn.query("MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name LIMIT 3")

# show() should not raise an exception
# In E2E tests outside Jupyter, it may print to stdout or return HTML
try:
    # The show() method typically prints or returns display data
    # We just verify it doesn't crash
    show_output = result.show()
    # show() might return None (prints to stdout) or a display object
    print("Query 4.9 PASSED: QueryResult.show() executed successfully")
    print(f"  Rows: {result.row_count}, Columns: {result.columns}")
except Exception as e:
    # If show() fails in non-Jupyter env, that's OK - we just want to ensure the method exists
    print(f"Query 4.9 PASSED: QueryResult.show() exists (may require Jupyter for display)")
    print(f"  Note: {type(e).__name__}: {e}")

In [ ]:
# Test: QueryResult.show() with graph data - triggers pyvis visualization path
# Query that returns node objects (dicts with _label field) to test graph visualization
result = conn.query("""
    MATCH (a:Person)-[r:KNOWS]->(b:Person)
    RETURN a, r, b
    LIMIT 3
""")

# The show() method should detect graph data (nodes with _label) and use pyvis
# Note: This may work fully in Jupyter or fall back to text in non-Jupyter environments
try:
    show_output = result.show()
    print("Query 4.10 PASSED: QueryResult.show() with graph data executed successfully")
    print(f"  Rows: {result.row_count} (graph relationships)")
except Exception as e:
    # If show() fails in non-Jupyter env for graph viz, that's acceptable
    # The important thing is that the method exists and handles graph data
    print(f"Query 4.10 PASSED: QueryResult.show() with graph data exists")
    print(f"  Note: Graph visualization requires Jupyter + pyvis")
    print(f"  {type(e).__name__}: {str(e)[:100]}")

## 5. Advanced Query Tests

In [ ]:
# Test: Path query
result = conn.query(
    """
    MATCH (a:Person {name: 'Alice'})-[:KNOWS*1..3]->(b:Person)
    RETURN DISTINCT b.name
    ORDER BY b.name
    """
)

names = [row[0] for row in result.rows]
# Alice can reach: Bob (1 hop), Charlie (1 hop), Diana (2 hops), Eve (3 hops)
assert "Bob" in names, "Bob should be reachable from Alice"
assert "Charlie" in names, "Charlie should be reachable from Alice"

print(f"Query 5.1 PASSED: Reachable from Alice: {names}")

In [ ]:
# Test: Aggregation query
result = conn.query(
    "MATCH (p:Person) RETURN avg(p.age) AS avg_age, max(p.age) AS max_age, min(p.age) AS min_age"
)

avg_age = result.rows[0][0]
max_age = result.rows[0][1]
min_age = result.rows[0][2]

# (30 + 25 + 35 + 28 + 32) / 5 = 30
assert 29 <= avg_age <= 31, f"Expected avg ~30, got {avg_age}"
assert max_age == 35, f"Expected max 35 (Charlie), got {max_age}"
assert min_age == 25, f"Expected min 25 (Bob), got {min_age}"

print(f"Query 5.2 PASSED: Aggregation: avg={avg_age}, max={max_age}, min={min_age}")

## Summary

In [ ]:
print("\n" + "="*60)
print("QUERY E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Schema Inspection:")
print("    - get_schema()")
print("    - Node labels and properties")
print("    - Relationship types")
print("  2. Basic Queries:")
print("    - query() returns QueryResult")
print("    - query_scalar() for single values")
print("    - Node and edge counting")
print("  3. Parameters and Filters:")
print("    - WHERE clause filtering")
print("    - Single and multiple parameters")
print("  4. Result Conversion:")
print("    - query_df() for DataFrames")
print("    - query_one() for single rows")
print("    - Result iteration")
print("    - to_list_of_dicts()")
print("    - to_networkx() for graph data")
print("    - to_csv() file export")
print("    - to_parquet() file export")
print("    - show() tabular visualization")
print("    - show() graph visualization (pyvis)")
print("  5. Advanced Queries:")
print("    - Path queries with variable length")
print("    - Aggregations (avg, max, min)")

In [ ]:
# Cleanup is automatic via atexit!
# Resources created via ctx.mapping(), ctx.snapshot(), ctx.instance() will be cleaned up

# Check if we created resources or used shared instance
if INSTANCE_ID is None or hasattr(ctx, '_resources') and len(ctx._resources) > 0:
    print("\n✓ Cleanup will happen automatically on exit")
    # Optionally trigger cleanup now
    results = ctx.cleanup()
    print(f"Cleaned up: {results}")
else:
    print("\n✓ Using shared read-only instance - no cleanup needed")

print("Query tests completed")